# BhramGuard URL Model Training

Train a phishing classifier for URLs from `backend/ml/datasets/urls.csv`.

This notebook starts with a URL baseline: character-level TF-IDF plus lexical/structural URL features. It saves model artifacts into `backend/ml/models`.

In [19]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[2]

DATASET_PATH = PROJECT_ROOT / "backend" / "ml" / "datasets" / "urls.csv"
MODEL_DIR = PROJECT_ROOT / "backend" / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)
print("Models:", MODEL_DIR)

Project root: c:\Users\Bidur\Desktop\BhramGuard-
Dataset: c:\Users\Bidur\Desktop\BhramGuard-\backend\ml\datasets\urls.csv
Models: c:\Users\Bidur\Desktop\BhramGuard-\backend\ml\models


In [20]:
import ipaddress
import re
from types import SimpleNamespace
from urllib.parse import urlparse

import joblib
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import MaxAbsScaler
from sklearn.svm import LinearSVC

In [21]:
df_url = pd.read_csv(DATASET_PATH)

print("Shape:", df_url.shape)
print("Columns:", df_url.columns.tolist())
df_url.head()

Shape: (835697, 2)
Columns: ['text', 'label']


,text,label
0,http://webmail-brinkster.com/ex/?email=%20%0%,1
1,billsportsmaps.com/?p=1206,0
2,www.sanelyurdu.com/language/homebank.tsbbank.c...,1
3,ee-billing.limited323.com,1
4,indiadaily.com/bolly_archive.htm,0


In [22]:
def infer_columns(df):
    label_candidates = ["label", "Label", "class", "target", "is_phishing"]
    url_candidates = ["url", "URL", "text", "link", "domain"]

    label_col = next((col for col in label_candidates if col in df.columns), None)
    if label_col is None:
        raise ValueError(f"No label column found. Columns: {df.columns.tolist()}")

    url_col = next((col for col in url_candidates if col in df.columns and col != label_col), None)
    if url_col is None:
        object_cols = [col for col in df.columns if col != label_col and df[col].dtype == "object"]
        if not object_cols:
            raise ValueError(f"No URL column found. Columns: {df.columns.tolist()}")
        url_col = object_cols[0]

    return url_col, label_col

URL_COL, LABEL_COL = infer_columns(df_url)
print("URL column:", URL_COL)
print("Label column:", LABEL_COL)
print(df_url[LABEL_COL].value_counts(dropna=False))

URL column: text
Label column: label
label
0    444933
1    390764
Name: count, dtype: int64


In [23]:
df_url = df_url[[URL_COL, LABEL_COL]].dropna()
df_url[URL_COL] = df_url[URL_COL].astype(str).str.strip()
df_url[LABEL_COL] = df_url[LABEL_COL].astype(int)
df_url = df_url.drop_duplicates(subset=[URL_COL]).reset_index(drop=True)

print("Cleaned shape:", df_url.shape)
print(df_url[LABEL_COL].value_counts(normalize=True))

Cleaned shape: (821399, 2)
label
0    0.529844
1    0.470156
Name: proportion, dtype: float64


In [24]:
def parse_url(value):
    url = "" if pd.isna(value) else str(value).strip()
    candidate = url if re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", url) else "http://" + url

    try:
        return url, urlparse(candidate), 0
    except ValueError:
        fallback = SimpleNamespace(scheme="", hostname="", path="", query="")
        return url, fallback, 1

def safe_url_part(parsed, part_name):
    try:
        return getattr(parsed, part_name) or ""
    except ValueError:
        return ""

def hostname_is_ip(hostname):
    if not hostname:
        return 0
    try:
        ipaddress.ip_address(hostname)
        return 1
    except ValueError:
        return 0

def extract_url_features(value):
    url, parsed, parse_error = parse_url(value)
    lowered = url.lower()
    hostname = safe_url_part(parsed, "hostname")
    path = safe_url_part(parsed, "path")
    query = safe_url_part(parsed, "query")
    suspicious_words = ["login", "verify", "secure", "account", "update", "confirm", "bank", "wallet", "password"]

    return {
        "url_length": len(url),
        "hostname_length": len(hostname),
        "path_length": len(path),
        "query_length": len(query),
        "dot_count": url.count("."),
        "hyphen_count": url.count("-"),
        "slash_count": url.count("/"),
        "question_count": url.count("?"),
        "equals_count": url.count("="),
        "at_count": url.count("@"),
        "percent_count": url.count("%"),
        "digit_count": sum(char.isdigit() for char in url),
        "has_https": int(parsed.scheme == "https"),
        "has_ip_hostname": hostname_is_ip(hostname),
        "subdomain_count": max(hostname.count(".") - 1, 0),
        "suspicious_word_count": sum(1 for word in suspicious_words if word in lowered),
        "parse_error": parse_error,
    }

manual_features = pd.DataFrame(df_url[URL_COL].apply(extract_url_features).tolist())
manual_features = manual_features.fillna(0)
print("Parse errors:", int(manual_features["parse_error"].sum()))
manual_features.describe().transpose()

Parse errors: 13


,count,mean,std,min,25%,50%,75%,max
url_length,821399.0,47.242701,42.199317,1.0,26.0,37.0,55.0,3992.0
hostname_length,821399.0,18.796881,11.799528,0.0,13.0,16.0,22.0,249.0
path_length,821399.0,20.574999,24.226577,0.0,1.0,14.0,31.0,2156.0
query_length,821399.0,5.692193,31.264335,0.0,0.0,0.0,0.0,3915.0
dot_count,821399.0,2.126629,1.493053,0.0,1.0,2.0,2.0,37.0
hyphen_count,821399.0,0.999140,2.199739,0.0,0.0,0.0,1.0,43.0
slash_count,821399.0,2.514756,1.863718,0.0,1.0,2.0,3.0,51.0
question_count,821399.0,0.128992,0.479168,0.0,0.0,0.0,0.0,195.0
equals_count,821399.0,0.217841,0.855029,0.0,0.0,0.0,0.0,78.0
at_count,821399.0,0.006119,0.083851,0.0,0.0,0.0,0.0,10.0


In [25]:
X_url_train, X_url_test, X_manual_train, X_manual_test, y_train, y_test = train_test_split(
    df_url[URL_COL],
    manual_features,
    df_url[LABEL_COL],
    test_size=0.2,
    random_state=42,
    stratify=df_url[LABEL_COL],
)

tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=5000,
    min_df=2,
    lowercase=True,
)

X_train_tfidf = tfidf.fit_transform(X_url_train)
X_test_tfidf = tfidf.transform(X_url_test)

manual_scaler = MaxAbsScaler()
X_manual_train_scaled = manual_scaler.fit_transform(X_manual_train)
X_manual_test_scaled = manual_scaler.transform(X_manual_test)

X_train = hstack([X_train_tfidf, csr_matrix(X_manual_train_scaled)])
X_test = hstack([X_test_tfidf, csr_matrix(X_manual_test_scaled)])

print("Train matrix:", X_train.shape)
print("Test matrix:", X_test.shape)

Train matrix: (657119, 5017)
Test matrix: (164280, 5017)


In [26]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced", solver="saga", n_jobs=-1),
    "linear_svc": LinearSVC(class_weight="balanced", random_state=42),
    "sgd_log_loss": SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42, n_jobs=-1),
    "complement_nb": ComplementNB(),
}

results = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    auc = roc_auc_score(y_test, probabilities) if probabilities is not None else None

    print("\n===", model_name, "===")
    print(classification_report(y_test, predictions))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))
    if auc is not None:
        print("ROC-AUC:", auc)

    report = classification_report(y_test, predictions, output_dict=True)
    results[model_name] = {"model": model, "auc": auc, "f1": report["weighted avg"]["f1-score"]}

best_model_name = max(results, key=lambda name: (results[name]["auc"] or 0, results[name]["f1"]))
best_model = results[best_model_name]["model"]
print("Best model:", best_model_name)


=== logistic_regression ===
              precision    recall  f1-score   support

           0       0.96      0.97      0.96     87043
           1       0.96      0.96      0.96     77237

    accuracy                           0.96    164280
   macro avg       0.96      0.96      0.96    164280
weighted avg       0.96      0.96      0.96    164280

Confusion matrix:
[[84141  2902]
 [ 3439 73798]]
ROC-AUC: 0.9922401137719716

=== linear_svc ===
              precision    recall  f1-score   support

           0       0.96      0.97      0.97     87043
           1       0.96      0.96      0.96     77237

    accuracy                           0.96    164280
   macro avg       0.96      0.96      0.96    164280
weighted avg       0.96      0.96      0.96    164280

Confusion matrix:
[[84311  2732]
 [ 3218 74019]]

=== sgd_log_loss ===
              precision    recall  f1-score   support

           0       0.93      0.96      0.94     87043
           1       0.95      0.92      0

In [27]:
artifact = {
    "model": best_model,
    "vectorizer": tfidf,
    "manual_scaler": manual_scaler,
    "manual_feature_columns": manual_features.columns.tolist(),
    "url_column": URL_COL,
    "label_column": LABEL_COL,
    "best_model_name": best_model_name,
}

joblib.dump(artifact, MODEL_DIR / "url_phishing_model.pkl")
print("Saved:", MODEL_DIR / "url_phishing_model.pkl")

Saved: c:\Users\Bidur\Desktop\BhramGuard-\backend\ml\models\url_phishing_model.pkl
